# SB100 Squad 2 — extração em lote

Extrai os PDFs **ainda não extraídos** e devolve o resultado ao Supabase.

Antes de começar:

1. **Ambiente de execução → Alterar tipo → GPU**
2. No ícone de chave (Secrets), crie três segredos com *Notebook access* ligado:
   `SUPABASE_URL`, `SUPABASE_ANON_KEY`, `SUPABASE_SERVICE_ROLE_KEY`

Se a sessão cair no meio, nada do que já subiu se perde: reabra e rode de novo,
que a seleção é por `extracted = false` e ele traz só o que falta.


## 1. Preparo


In [ ]:
from google.colab import userdata
from pathlib import Path
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU - troque em Ambiente de execucao")

!rm -rf /content/repo
!git clone -q https://github.com/nicolasaws1/parser_rag_framework.git /content/repo
%cd /content/repo
!pip install -q supabase python-dotenv pymupdf

# as chaves ficam nos Secrets, não no notebook: assim da para compartilhar
# este arquivo sem vazar a service_role
CHAVES = ("SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY")
with open("/content/repo/.env", "w") as f:
    for k in CHAVES:
        v = userdata.get(k)
        assert v, f"falta o segredo {k} (icone de chave na barra lateral)"
        print(f"{k}={v}", file=f)
print("ambiente pronto")


## 2. Ver o que falta

Só relata, não baixa nada.


In [ ]:
!python scripts/acervo.py 2>/dev/null | head -12
!python extractor/baixar_lote.py --quantos 0


## 3. Baixar o lote

Prioriza o que foi pedido pelo botão **Analisar** do site; depois os menores,
que dão retorno rápido e cabem numa sessão sem risco de perder tudo no meio.

Ele imprime a estimativa de GPU antes de você seguir.


In [ ]:
QUANTOS = 10   #@param {type:"integer"}
MENORES = True #@param {type:"boolean"}

extra = "--menores" if MENORES else ""
!python extractor/baixar_lote.py --quantos {QUANTOS} {extra}


## 4. Extrair

Instala YOLO, Docling e Chandra na primeira vez (uns 5 minutos), depois
processa tudo que está em `/content/pdfs`.

Se a sessão cair, rode esta célula de novo: ela pula o que já terminou.


In [ ]:
!python extractor/extrator_colab.py


## 5. Mandar para o Supabase

`ingerir_extracao.py` **atualiza** o documento que já existe e não encosta em
`article_metadata`. Não troque pelo `ingest_supabase.py`: aquele apaga a linha
de `pdfs` e recria, e o cascade levaria junto os metadados vindos da curadoria,
que aqui não há de onde repor.

`gerar_figuras.py` recorta cada gráfico do PDF pela bbox do banco. É
idempotente, então só faz o que falta.


In [ ]:
!python scripts/ingerir_extracao.py /content/export
!python scripts/gerar_figuras.py --aplicar


## 6. Conferir

Depois disto, abra o site: os documentos do lote aparecem como extraídos, com
as figuras.


In [ ]:
!python scripts/acervo.py 2>/dev/null | head -12


## 7. Guardar uma cópia (opcional)

O resultado já está no Supabase. Isto é só para ter o arquivo bruto da corrida
na sua máquina.


In [ ]:
import shutil
shutil.make_archive('/content/extracao_lote', 'zip', '/content/export')
from google.colab import files
files.download('/content/extracao_lote.zip')
